# UK Parliament Immigration Debates: Data Collection (2023–2025)

This notebook collects speeches from the UK House of Commons (2023–2025) 
that contain immigration-related keywords, labelled by party (Conservative / Labour).

**Keywords searched:** `immigration`, `migrant`, `asylum`  
**Data source:** [mySociety ParlParse XML](http://www.theyworkforyou.com/pwdata/scrapedxml/debates/)  
**Party data source:** [UK Parliament Members API](https://members-api.parliament.uk)

---
### Notebook Structure
1. Install & import libraries
2. Fetch MP → Party mapping
3. Discover available XML debate files
4. Download & parse XML files with keyword filter
5. Match speakers to parties
6. Manual party corrections for missing MPs
7. Save final dataset to CSV

## 1. Install & Import Libraries

In [1]:
# Uncomment the line below if packages are not yet installed
# !pip install requests pandas beautifulsoup4 lxml

import requests
import pandas as pd
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import re
import time
import json
from collections import Counter

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


## 2. Fetch MP → Party Mapping

We use the official UK Parliament Members API to build a dictionary that maps
each MP's display name to their party. We fetch both current and former MPs.

In [2]:
def fetch_party_map():
    """
    Fetches all MPs (current and former) from the Parliament Members API.
    Returns a dictionary: {display_name: party_name}
    """
    party_map = {}
    base_url = "https://members-api.parliament.uk/api/Members/Search"

    for is_current in [True, False]:
        skip = 0
        take = 100
        label = "current" if is_current else "former"
        print(f"  Fetching {label} MPs...")

        while True:
            params = {
                "House": 1,                              # House of Commons
                "IsCurrentMember": str(is_current).lower(),
                "skip": skip,
                "take": take
            }
            r = requests.get(base_url, params=params, timeout=15)
            data = r.json()
            items = data.get("items", [])
            if not items:
                break

            for item in items:
                v = item["value"]
                name_display = v.get("nameDisplayAs", "")
                party = v.get("latestParty", {}).get("name", "Unknown")
                if name_display:
                    party_map[name_display] = party

            skip += take
            total = data.get("totalResults", 0)
            if skip >= total:
                break
            time.sleep(0.2)

    return party_map


print("👥 Fetching MP–Party mapping from Parliament API...")
party_map = fetch_party_map()
print(f"\n✅ {len(party_map)} MPs loaded.")

# Quick check on party distribution
party_counts = Counter(party_map.values())
print("\nTop 5 parties:")
for party, count in party_counts.most_common(5):
    print(f"  {party}: {count}")

👥 Fetching MP–Party mapping from Parliament API...
  Fetching current MPs...
  Fetching former MPs...

✅ 500 MPs loaded.

Top 5 parties:
  Labour: 205
  Conservative: 187
  Liberal Democrat: 39
  Independent: 15
  Labour (Co-op): 15


## 3. Discover Available XML Debate Files (2023–2025)

ParlParse stores each day's debates as an XML file.
Files labelled `a` are redirect/alias files — we skip them and use `b`, `c`, `d`, etc.

In [3]:
BASE_URL = "http://www.theyworkforyou.com/pwdata/scrapedxml/debates/"

print("📋 Fetching list of debate XML files...")
r_index = requests.get(BASE_URL, timeout=10)
soup_index = BeautifulSoup(r_index.text, "lxml")

# Collect filenames for 2023–2025, skipping 'a' (redirect) files
all_files = [
    a["href"]
    for a in soup_index.find_all("a", href=True)
    if re.match(r"debates(202[3-5])-\d{2}-\d{2}[b-z]\.xml", a["href"])
]

print(f"✅ {len(all_files)} XML files found (2023–2025)")
print(f"  First 5 : {all_files[:5]}")
print(f"  Last  5 : {all_files[-5:]}")

📋 Fetching list of debate XML files...
✅ 839 XML files found (2023–2025)
  First 5 : ['debates2023-01-09b.xml', 'debates2023-01-10b.xml', 'debates2023-01-11b.xml', 'debates2023-01-11c.xml', 'debates2023-01-11d.xml']
  Last  5 : ['debates2025-12-16d.xml', 'debates2025-12-17b.xml', 'debates2025-12-18b.xml', 'debates2025-12-18c.xml', 'debates2025-12-18d.xml']


## 4. Download & Parse XML Files with Keyword Filter

For each file we:
1. Download the XML
2. Strip DOCTYPE/entity declarations that break Python's XML parser
3. Extract each `<speech>` element (speaker name + text)
4. Keep only speeches containing at least one keyword

In [4]:
KEYWORDS = ["immigration", "migrant", "asylum"]
KEYWORD_PATTERN = "|".join(KEYWORDS)


def parse_xml_content(xml_text, filename):
    """
    Parses a ParlParse XML debate file and returns a list of speech records.
    Entity declarations are stripped before parsing to avoid XML errors.
    """
    # Remove DOCTYPE and ENTITY declarations that Python's parser cannot handle
    xml_clean = re.sub(r'<!ENTITY[^>]+>', '', xml_text)
    xml_clean = re.sub(r'<!DOCTYPE[^[]*\[[^\]]*\]>', '', xml_clean)

    records = []
    try:
        root = ET.fromstring(xml_clean)
        for speech in root.iter("speech"):
            speaker = speech.get("speakername", "")

            # Extract date from filename, e.g. debates2023-01-10b.xml → 2023-01-10
            date_match = re.search(r"(\d{4}-\d{2}-\d{2})", filename)
            date = date_match.group(1) if date_match else ""

            # Concatenate all paragraph texts
            paragraphs = [p.text for p in speech.findall("p") if p.text]
            text = " ".join(paragraphs).strip()

            if text and speaker:
                records.append({
                    "date":     date,
                    "speaker":  speaker,
                    "text":     text,
                    "filename": filename
                })
    except ET.ParseError:
        pass  # Skip malformed files silently

    return records


# ── Main download loop ──────────────────────────────────────
all_records = []
errors = 0

print(f"🔍 Scanning {len(all_files)} XML files — this takes ~10–15 minutes...\n")

for i, filename in enumerate(all_files):
    url = BASE_URL + filename
    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            continue

        records = parse_xml_content(r.text, filename)

        # Keep only speeches that contain at least one keyword
        for rec in records:
            if re.search(KEYWORD_PATTERN, rec["text"], re.IGNORECASE):
                all_records.append(rec)

    except Exception:
        errors += 1

    # Progress report every 50 files
    if (i + 1) % 50 == 0:
        print(f"  📊 {i+1}/{len(all_files)} files | {len(all_records)} keyword matches so far")

    time.sleep(0.1)  # Be polite to the server

print(f"\n✅ Scan complete!")
print(f"   Total keyword matches : {len(all_records)}")
print(f"   Files with errors     : {errors}")

🔍 Scanning 839 XML files — this takes ~10–15 minutes...

  📊 50/839 files | 122 keyword matches so far
  📊 100/839 files | 601 keyword matches so far
  📊 150/839 files | 966 keyword matches so far
  📊 200/839 files | 1183 keyword matches so far
  📊 250/839 files | 1647 keyword matches so far
  📊 300/839 files | 1815 keyword matches so far
  📊 350/839 files | 1955 keyword matches so far
  📊 400/839 files | 2101 keyword matches so far
  📊 450/839 files | 2161 keyword matches so far
  📊 500/839 files | 2276 keyword matches so far
  📊 550/839 files | 2393 keyword matches so far
  📊 600/839 files | 2546 keyword matches so far
  📊 650/839 files | 2922 keyword matches so far
  📊 700/839 files | 3016 keyword matches so far
  📊 750/839 files | 3239 keyword matches so far
  📊 800/839 files | 3499 keyword matches so far

✅ Scan complete!
   Total keyword matches : 3549
   Files with errors     : 0


## 5. Match Speakers to Parties

We map each speaker name to a party using `party_map`.
Unknown speakers are queried individually via the Parliament API.

In [5]:
df = pd.DataFrame(all_records)

# First pass: use the pre-built party_map
df["party"] = df["speaker"].map(party_map).fillna("Unknown")
df["party"] = df["party"].replace("Labour (Co-op)", "Labour")

# Second pass: query unknown speakers one by one via the API
unknown_speakers = df[df["party"] == "Unknown"]["speaker"].unique()
print(f"🔍 Querying {len(unknown_speakers)} unknown speakers individually...")

extra_map = {}
for i, name in enumerate(unknown_speakers):
    try:
        r = requests.get(
            "https://members-api.parliament.uk/api/Members/Search",
            params={"Name": name, "House": 1},
            timeout=10
        )
        items = r.json().get("items", [])
        if items:
            v = items[0]["value"]
            extra_map[name] = v.get("latestParty", {}).get("name", "Unknown")
    except Exception:
        pass

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(unknown_speakers)} queried...")

    time.sleep(0.2)

party_map.update(extra_map)

# Re-apply updated map
df["party"] = df["speaker"].map(party_map).fillna("Unknown")
df["party"] = df["party"].replace("Labour (Co-op)", "Labour")

print(f"\n✅ After individual lookup:")
print(f"   Still Unknown : {(df['party'] == 'Unknown').sum()}")

🔍 Querying 507 unknown speakers individually...
  50/507 queried...
  100/507 queried...
  150/507 queried...
  200/507 queried...
  250/507 queried...
  300/507 queried...
  350/507 queried...
  400/507 queried...
  450/507 queried...
  500/507 queried...

✅ After individual lookup:
   Still Unknown : 198


## 6. Manual Party Corrections

A small number of MPs are not returned by the API (e.g. retired members,
the Speaker, or those with name variations). We assign their parties manually.

In [6]:
# Manually confirmed party assignments
manual_map = {
    "Diana R. Johnson":       "Labour",
    "Stuart McDonald":        "Scottish National Party",
    "Bill Cash":              "Conservative",
    "John Martin McDonnell":  "Labour",
    "Theresa May":            "Conservative",
    "Rosie Winterton":        "Labour",
    "Thangam Debbonaire":     "Labour",
    "Andrew Slaughter":       "Labour",
    "Tan Dhesi":              "Labour",
    "Liz Saville-Roberts":    "Plaid Cymru",
    "John McDonnell":         "Labour",
    "Caroline Nokes":         "Conservative",
    "Harriet Harman":         "Labour",
    "Jeremy Corbyn":          "Independent",
    "Damian Green":           "Conservative",
    "David Davis":            "Conservative",
    "Alistair Carmichael":    "Liberal Democrat",
    "Paul Blomfield":         "Labour",
    "Bambos Charalambous":    "Labour",
}

party_map.update(manual_map)

# Final party assignment
df["party"] = df["speaker"].map(party_map).fillna("Unknown")
df["party"] = df["party"].replace("Labour (Co-op)", "Labour")

print(f"✅ Manual corrections applied.")
print(f"   Still Unknown : {(df['party'] == 'Unknown').sum()}")

✅ Manual corrections applied.
   Still Unknown : 56


## 7. Build Final Dataset & Save to CSV

We keep only Conservative and Labour speeches, add keyword indicator columns,
and save everything to a CSV file ready for sentiment analysis.

In [7]:
# Keep only Conservative and Labour
df_final = df[df["party"].isin(["Conservative", "Labour"])].copy()

# Add a boolean column for each keyword
for kw in KEYWORDS:
    df_final[f"has_{kw}"] = df_final["text"].str.contains(kw, case=False)

# Parse date and extract year
df_final["date"] = pd.to_datetime(df_final["date"])
df_final["year"] = df_final["date"].dt.year

# ── Summary report ─────────────────────────────────────────
print("=" * 55)
print("📊 FINAL DATASET SUMMARY")
print("=" * 55)
print(f"\nTotal speeches (Conservative + Labour): {len(df_final)}")

print(f"\n{'Party':<20} {'Speeches':>10}")
print("-" * 32)
for party, count in df_final.groupby("party").size().items():
    print(f"{party:<20} {count:>10}")

print(f"\n{'Keyword':<20} {'Conservative':>15} {'Labour':>10}")
print("-" * 47)
for kw in KEYWORDS:
    con = df_final[df_final["party"] == "Conservative"][f"has_{kw}"].sum()
    lab = df_final[df_final["party"] == "Labour"][f"has_{kw}"].sum()
    print(f"{kw:<20} {con:>15} {lab:>10}")

print(f"\n{'Year':<10} {'Conservative':>15} {'Labour':>10}")
print("-" * 37)
for year in sorted(df_final["year"].unique()):
    con = len(df_final[(df_final["year"] == year) & (df_final["party"] == "Conservative")])
    lab = len(df_final[(df_final["year"] == year) & (df_final["party"] == "Labour")])
    print(f"{year:<10} {con:>15} {lab:>10}")

# ── Save ───────────────────────────────────────────────────
output_file = "hansard_immigration_2023_2025.csv"
df_final.to_csv(output_file, index=False)
print(f"\n💾 Saved: {output_file}")
print(f"   Rows: {len(df_final)} | Columns: {list(df_final.columns)}")
df_final.head(3)

📊 FINAL DATASET SUMMARY

Total speeches (Conservative + Labour): 2468

Party                  Speeches
--------------------------------
Conservative               1084
Labour                     1384

Keyword                 Conservative     Labour
-----------------------------------------------
immigration                      522        650
migrant                          351        223
asylum                           532        890

Year          Conservative     Labour
-------------------------------------
2023                   476        347
2024                   317        368
2025                   291        669

💾 Saved: hansard_immigration_2023_2025.csv
   Rows: 2468 | Columns: ['date', 'speaker', 'text', 'filename', 'party', 'has_immigration', 'has_migrant', 'has_asylum', 'year']


,date,speaker,text,filename,party,has_immigration,has_migrant,has_asylum,year
1,2023-01-10,Maggie Throup,What discussions he has had with Cabinet colle...,debates2023-01-10b.xml,Conservative,True,False,False,2023
2,2023-01-10,Dean Russell,What discussions he has had with Cabinet colle...,debates2023-01-10b.xml,Conservative,True,False,False,2023
3,2023-01-10,Dominic Raab,We are committed to ensuring faster throughput...,debates2023-01-10b.xml,Conservative,True,False,True,2023
